In [ ]:
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import requests
from datasets import load_dataset
from IPython.display import HTML, display
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

REPO_ROOT = Path("/lfs/skampere2/0/animjha/CleanSlate")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SCRIPTS_DIR = REPO_ROOT / "scripts" / "dataset_search"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from curation.retrieval.backend_infini_gram import load_engine

from igm_lib import (
    WORD_RE,
    WordNgramMatch,
    compute_matched_word_ngrams,
    per_word_occ_from_matches,
    plot_word_occ_strips,
    retrieve_docs,
    top_spans_by_length,
    unique_docs_for_spans,
    word_spans,
)


In [ ]:
df = load_dataset("unlearning-cleanslate/cleanslate_dataset", "default")["train"].to_polars()
df_olmo = load_dataset("unlearning-cleanslate/verif-olmo-3-1125-32b-w100-s10","default")["train"].to_polars()
IRIS_ID = "9bd7dd3ff9165c9ecce1313aed6a2915fd8f43ab"
NEVER_GONNA_ID = "b9bb555ab4913dfeaafb409555e8f543f5180fc6"
ASOIAF_ID = "731bc81c4a7cd705342bb30f8b6d9f4a5d658d04"
ACTIVE_ID = NEVER_GONNA_ID
reference_text = df_olmo.filter(pl.col("song_id") == ACTIVE_ID)["reference_target"].item()
MIN_N_GRAM = 3
MIN_COUNT = 0

In [10]:
top_ids = df.with_columns(pl.col("reference_target").str.len_chars().alias("ref_len")).sort("ref_len", descending=True).head(5)["content_id"].to_list()


In [11]:
def content_title_from_id(song_id:str)-> str:
    title =  df_olmo.filter(pl.col("song_id") == song_id)["song_title"].item()
    creators = df_olmo.filter(pl.col("song_id") == song_id)["content_creators"].item()
    year = df_olmo.filter(pl.col("song_id") == song_id)["content_year"].item()
    return f"{title} - {creators} ({year})"

In [12]:
for id in top_ids:
    print(content_title_from_id(id),id)

A Song of Ice and Fire (Complete) - George R.R. Martin (1996) 731bc81c4a7cd705342bb30f8b6d9f4a5d658d04
The Complete Novels of George Orwell - George Orwell (1949) 225cde9fb93963bee1a264bcc6d296b6361374e0
War and Peace - Leo Tolstoy (1869) 6badee68f339d8e8b416b69619ef8b80d56f42b5
The Lord of the Rings - J.R.R. Tolkien (1954) 48d98b0f0b7d5bc7bc9499c798a5c337af37afc9
The Maze Runner Series - James Dashner (2009) 4d50a24058da021d176e81365370ffadbf06a6ed


In [4]:
LOCAL_INDEX_DIR = REPO_ROOT /"curation/indices/10B/infini_gram_mini/b6c7e0c77ec6"                      
engine = load_engine(LOCAL_INDEX_DIR, load_to_ram=True)
print(f"loaded engine from {LOCAL_INDEX_DIR}")    

loaded engine from /lfs/skampere2/0/animjha/CleanSlate/curation/indices/10B/infini_gram_mini/b6c7e0c77ec6


In [5]:
# WordNgramMatch / word_spans / compute_matched_word_ngrams now live in igm_lib.
# These two helpers stay here because they're notebook-only debug utilities.


def word_match_counts(matches: list[WordNgramMatch], n_words: int) -> np.ndarray:
    """Per word: number of matched ngrams containing that word."""
    v = np.zeros(n_words, dtype=np.int32)
    for m in matches:
        v[m.word_start:m.word_end] += 1
    return v


def char_aligned_word_counts(
    matches: list[WordNgramMatch],
    spans: list[tuple[int, int]],
    n_chars: int,
) -> np.ndarray:
    """Char strip: each word's chars get that word's match count; gaps stay 0."""
    counts = word_match_counts(matches, len(spans))
    v = np.zeros(n_chars, dtype=np.int32)
    for (cs, ce), c in zip(spans, counts):
        v[cs:ce] = c
    return v


In [6]:
matches = compute_matched_word_ngrams(reference_text, engine, min_n=MIN_N_GRAM, min_count=MIN_COUNT)
print(f"{len(matches)} matched ngrams from {len(word_spans(reference_text))} words")

3715 matched ngrams from 362 words


In [7]:
N_VALUES = [n for n in [5, 7, 10, 15, 25, 50] if n >= MIN_N_GRAM]

occ_counts_by_n: dict[int, np.ndarray] = {}
for n in N_VALUES:
    occ = per_word_occ_from_matches(reference_text, matches, n)
    occ_counts_by_n[n] = occ
    print(f"N={n}: occ_max={occ.max()}, n_words_lit={(occ > 0).sum()} / {len(occ)}")


N=5: occ_max=178, n_words_lit=341 / 362
N=7: occ_max=91, n_words_lit=326 / 362
N=10: occ_max=60, n_words_lit=288 / 362
N=15: occ_max=87, n_words_lit=222 / 362
N=25: occ_max=52, n_words_lit=204 / 362
N=50: occ_max=0, n_words_lit=0 / 362


In [ ]:
plot_word_occ_strips(reference_text, occ_counts_by_n, title=content_title_from_id(ACTIVE_ID))
plt.show()


In [9]:
TOP_K_SPANS = 20000
DOCS_PER_SPAN = 10
RETRIEVE_WORKERS = 128

selected_spans = top_spans_by_length(matches, TOP_K_SPANS)
length_lo = selected_spans[-1].word_end - selected_spans[-1].word_start
length_hi = selected_spans[0].word_end - selected_spans[0].word_start
print(f"selected {len(selected_spans)} spans (word length range: {length_lo}..{length_hi})")

unique_docs = unique_docs_for_spans(
    reference_text, selected_spans, engine,
    docs_per_span=DOCS_PER_SPAN, workers=RETRIEVE_WORKERS,
)
print(f"{len(unique_docs)} unique docs across the top {TOP_K_SPANS} spans (D={DOCS_PER_SPAN})")

docs_df = pl.DataFrame([
    {"doc_ix": d["doc_ix"], "path": d["path"], "n_unique_source_spans": len(d["unique_source_spans"])}
    for d in unique_docs
]).sort("n_unique_source_spans", descending=True)
docs_df


selected 3715 spans (word length range: 3..34)


retrieve_unique_docs_spans: 100%|██████████| 3715/3715 [00:10<00:00, 360.84it/s]

761 unique docs across the top 20000 spans (D=10)


doc_ix,path,n_unique_source_spans
i64,str,i64
22752506,"""stack_edu_fim-Python_0018/shar…",769
10474288,"""stack_edu_fim-Cpp_0016/shard_0…",174
10905717,"""stack_edu_fim-Python_0017/shar…",117
4399615,"""general_reasoning_mix/shard_00…",64
10017717,"""reddit_to_flashcards/lowthresh…",51
…,…,…
4934718,"""nemotron-synth-qa/shard_000008…",1
5015760,"""nemotron-synth-qa/shard_000008…",1
500681,"""common_crawl-high-quality_20_e…",1
